## Environment Setup and Authentication

In [1]:
import sys
import os

# SmolLM3 requires a recent Transformers release (>=4.53.0) plus the rest of the HF stack
!{sys.executable} -m pip install --upgrade --quiet "transformers>=4.53.0" accelerate huggingface_hub datasets

import torch
import math
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | Detected {device_count} GPUs.")

# Authentication (SmolLM3-3B is a public/ungated model, but we keep the same
# Kaggle-secrets flow used for LLaMA so the notebooks stay consistent)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = ""
    print("No HF_TOKEN secret found — continuing without authentication (SmolLM3-3B is a public model).")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
PyTorch 2.10.0+cu128 | Detected 2 GPUs.
No HF_TOKEN secret found — continuing without authentication (SmolLM3-3B is a public model).


## Evaluation Metrics

In [2]:
class IntrinsicEvaluator:
    def __init__(self, model, tokenizer, device="cuda:0"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_lengths(self, text: str):
        """Calculates text-intrinsic properties for normalization."""
        if not isinstance(text, str):
            text = str(text)
        num_chars = max(1, len(text))
        num_bytes = max(1, len(text.encode('utf-8')))
        num_words = max(1, len(text.strip().split()))
        return num_chars, num_bytes, num_words

    def evaluate_sequence(self, text: str):
        """Calculates raw NLL and normalizes it into BPB, BPC, and BPW."""
        chars, bytes_count, words = self.get_lengths(text)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            # HF CausalLM loss is the average negative log-likelihood per token
            seq_len = inputs["input_ids"].shape[1]
            neg_log_likelihood = outputs.loss.item() * (seq_len - 1)

        # Calculate Tokenizer-Independent Metrics
        bpb = neg_log_likelihood / (bytes_count * math.log(2))
        bpc = neg_log_likelihood / (chars * math.log(2))
        bpw = neg_log_likelihood / (words * math.log(2))

        return {
            "nll": neg_log_likelihood,
            "bpb": bpb,
            "bpc": bpc,
            "bpw": bpw
        }

print("IntrinsicEvaluator class defined.")


IntrinsicEvaluator class defined.


## Standardized Data Loading & Preprocessing

In [3]:
diverse_sentences_path = "/kaggle/input/datasets/uom220165f/eshinmenusha/diverse_sentences (1).csv"
sinhala_mixed_path = "/kaggle/input/datasets/uom220165f/eshinmenusha/sinhala_mixed_dataset.csv"

try:
    diverse_df = pd.read_csv(diverse_sentences_path).dropna()
    mixed_df = pd.read_csv(sinhala_mixed_path).dropna()
    print(f"Loaded Diverse Sentences: {len(diverse_df)} parallel pairs.")
    print(f"Loaded Mixed Script: {len(mixed_df)} sentences.")
except Exception as e:
    print(f"Data loading failed. Check your file paths. Error: {e}")


Loaded Diverse Sentences: 500 parallel pairs.
Loaded Mixed Script: 500 sentences.


## Model & Tokenizer Initialization

In [4]:
model_id = "HuggingFaceTB/SmolLM3-3B"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN if HF_TOKEN else None)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model natively in fp16 (single GPU is plenty for a 3B model on Kaggle)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN if HF_TOKEN else None,
    low_cpu_mem_usage=True
)
model.eval()

# Initialize our custom evaluator
evaluator = IntrinsicEvaluator(model, tokenizer)

print(f"Model loaded.")
print(f"Precision: {model.dtype} | Device Map: {model.hf_device_map}")


Loading HuggingFaceTB/SmolLM3-3B...


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Model loaded.
Precision: torch.float16 | Device Map: {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1}


## Pilot Run

In [5]:
import csv

pilot_output_path = 'pilot_intrinsic_results.csv'
print("Starting Pilot Run (First 20 items)...")

# Open CSV in append mode to protect against runtime crashes
with open(pilot_output_path, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    # Write standardized header
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw'
    ])

    # Slice the first 20 rows of our loaded Kaggle dataframe
    pilot_sample = diverse_df.head(20)

    for index, row in pilot_sample.iterrows():
        # 1. Evaluate Native Unicode
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])

        # 2. Evaluate Romanized (Singlish)
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        # Log to disk
        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw']
        ])

        if (index + 1) % 5 == 0:
            print(f"Processed {index + 1}/20 parallel pairs...")

print(f"\n Pilot run complete. Results saved to {pilot_output_path}")

# Display verification preview
preview_df = pd.read_csv(pilot_output_path)
print("\nFirst 3 rows of calculated metrics:")
display(preview_df.head(3))


Starting Pilot Run (First 20 items)...
Processed 5/20 parallel pairs...
Processed 10/20 parallel pairs...
Processed 15/20 parallel pairs...
Processed 20/20 parallel pairs...

 Pilot run complete. Results saved to pilot_intrinsic_results.csv

First 3 rows of calculated metrics:


,item_index,unicode_nll,unicode_bpb,unicode_bpc,unicode_bpw,romanized_nll,romanized_bpb,romanized_bpc,romanized_bpw
0,0,91.576218,1.693802,4.403885,22.019426,119.807970,4.548589,4.548589,28.807727
1,1,83.120026,1.816922,4.996535,29.979212,91.848115,4.732458,4.732458,33.127205
2,2,97.554635,1.695682,4.540051,23.456931,110.624374,4.433257,4.433257,26.599539


## Full Intrinsic Evaluation

In [6]:
import csv
import pandas as pd
from tqdm import tqdm

# Evaluate Parallel Diverse Sentences (Unicode vs Romanized)
full_diverse_output = 'smollm3_3b_diverse_intrinsic.csv'
print(f"Starting full parallel evaluation on {len(diverse_df)} items...")

with open(full_diverse_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw'
    ])

    for index, row in tqdm(diverse_df.iterrows(), total=len(diverse_df), desc="Processing Parallel Pairs"):
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw']
        ])

print(f"Diverse evaluation complete. Saved to {full_diverse_output}")

# Evaluate Authentic Mixed-Script Sentences
full_mixed_output = 'smollm3_3b_mixed_intrinsic.csv'
mixed_text_col = 'text' if 'text' in mixed_df.columns else mixed_df.columns[0]
print(f"\nStarting full mixed-script evaluation on {len(mixed_df)} items (column: '{mixed_text_col}')...")

with open(full_mixed_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['item_index', 'mixed_nll', 'mixed_bpb', 'mixed_bpc', 'mixed_bpw'])

    for index, row in tqdm(mixed_df.iterrows(), total=len(mixed_df), desc="Processing Mixed Script"):
        mix_metrics = evaluator.evaluate_sequence(row[mixed_text_col])
        writer.writerow([
            index,
            mix_metrics['nll'], mix_metrics['bpb'], mix_metrics['bpc'], mix_metrics['bpw']
        ])

print(f"Mixed-script evaluation complete. Saved to {full_mixed_output}")


Starting full parallel evaluation on 500 items...


Processing Parallel Pairs: 100%|██████████| 500/500 [01:05<00:00,  7.60it/s]


Diverse evaluation complete. Saved to smollm3_3b_diverse_intrinsic.csv

Starting full mixed-script evaluation on 500 items (column: 'text')...


Processing Mixed Script: 100%|██████████| 500/500 [00:33<00:00, 15.12it/s]

Mixed-script evaluation complete. Saved to smollm3_3b_mixed_intrinsic.csv


In [7]:
# Aggregate and Display Unicode vs Romanized Summary
def calculate_full_aggregates(csv_path, model_label="SmolLM3-3B"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Unicode (Native)': [
            df['unicode_bpb'].mean(),
            df['unicode_bpc'].mean(),
            df['unicode_bpw'].mean()
        ],
        'Romanized (Singlish)': [
            df['romanized_bpb'].mean(),
            df['romanized_bpc'].mean(),
            df['romanized_bpw'].mean()
        ]
    }

    results_df = pd.DataFrame(summary)
    results_df['Degradation Factor'] = results_df['Romanized (Singlish)'] / results_df['Unicode (Native)']

    print("==========================================================")
    print(f"      {model_label} INTRINSIC EVALUATION SUMMARY")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Unicode (Native)': '{:.4f}',
        'Romanized (Singlish)': '{:.4f}',
        'Degradation Factor': '{:.2f}×'
    }).set_caption(f"{model_label} — Unicode vs Romanized Sinhala").background_gradient(
        subset=['Degradation Factor'], cmap='Reds'
    )
    display(styled)

    return results_df

full_summary = calculate_full_aggregates(full_diverse_output)

      SmolLM3-3B INTRINSIC EVALUATION SUMMARY
                  Metric  Unicode (Native)  Romanized (Singlish)  Degradation Factor
     Bits-Per-Byte (BPB)            1.9327                4.2304              2.1889
Bits-Per-Character (BPC)            5.1088                4.2397              0.8299
     Bits-Per-Word (BPW)           27.3229               26.1294              0.9563


,Metric,Unicode (Native),Romanized (Singlish),Degradation Factor
0,Bits-Per-Byte (BPB),1.9327,4.2304,2.19×
1,Bits-Per-Character (BPC),5.1088,4.2397,0.83×
2,Bits-Per-Word (BPW),27.3229,26.1294,0.96×


In [8]:
def calculate_mixed_aggregates(csv_path, model_label="SmolLM3-3B"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Mixed-Script': [
            df['mixed_bpb'].mean(),
            df['mixed_bpc'].mean(),
            df['mixed_bpw'].mean()
        ],
        'Std Dev': [
            df['mixed_bpb'].std(),
            df['mixed_bpc'].std(),
            df['mixed_bpw'].std()
        ]
    }

    results_df = pd.DataFrame(summary)

    print("==========================================================")
    print(f"      {model_label} MIXED-SCRIPT INTRINSIC SUMMARY")
    print(f"      ({len(df)} sentences)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Mixed-Script': '{:.4f}',
        'Std Dev': '{:.4f}'
    }).set_caption(f"{model_label} — Mixed-Script Sinhala")
    display(styled)

    return results_df

mixed_summary = calculate_mixed_aggregates(full_mixed_output)


      SmolLM3-3B MIXED-SCRIPT INTRINSIC SUMMARY
      (500 sentences)
                  Metric  Mixed-Script  Std Dev
     Bits-Per-Byte (BPB)        2.4531   0.5786
Bits-Per-Character (BPC)        4.6952   0.5866
     Bits-Per-Word (BPW)       28.5315   4.7114


,Metric,Mixed-Script,Std Dev
0,Bits-Per-Byte (BPB),2.4531,0.5786
1,Bits-Per-Character (BPC),4.6952,0.5866
2,Bits-Per-Word (BPW),28.5315,4.7114
